In [ ]:
!pip install tensorflow_addons

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow_addons as tfa
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Softmax, Lambda, Activation, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.nn import local_response_normalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau

from sklearn.model_selection import train_test_split

%matplotlib inline

In [ ]:
np.random.seed(2022)

# 1. Datasets

In [ ]:
cifair10 = tf.keras.datasets.cifar10.load_data()

(X_train, y_train), (X_test, y_test) = cifair10

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
X_test.shape

In [ ]:
y_train.min(), y_train.max()

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(15, 15))

for i, ax in enumerate(axes.flatten()):
    ax.axis('off')
    ax.imshow(X_train[i])

## Preprocess the dataset

In [ ]:
# One-hot encoding the labels
y_train = pd.get_dummies(y_train.squeeze()).values
y_test = pd.get_dummies(y_test.squeeze()).values

In [ ]:
y_train.shape

It's written in the article that:
"_Given a rectangular image, we first rescaled the image such that the shorter side was of length 256, and then cropped out the central 256?256 patch from the resulting image. We did not pre-process the images in any other way, except for subtracting the mean activity over the training set from each pixel. So we trained our network on the (centered) raw RGB values of the pixels._"

Since our images are already in the shape of 32x32 so we only subtract the mean activity over the training set from each pixel. See the next cell

In [ ]:
X_train = X_train - X_train.mean(axis=0)
X_test = X_test - X_test.mean(axis=0)

## Data Augmentation

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2)

In [ ]:
train_generator = ImageDataGenerator(
    rotation_range=2, 
    horizontal_flip=True,
    zoom_range=.1)

test_generator = ImageDataGenerator(
    rotation_range=2, 
    horizontal_flip=True,
    zoom_range=.1)

val_generator = ImageDataGenerator(
    rotation_range=2, 
    horizontal_flip=True,
    zoom_range=.1)



In [ ]:
train_generator.fit(X_train)
test_generator.fit(X_test)
val_generator.fit(X_val)

# 2. The Architecture

In [ ]:


model = Sequential([
    # 1st convolutional layer, 96 kernels of size 11?11?3 with a stride of 4 pixels
    Conv2D(filters=96, activation='relu', input_shape=(32, 32, 3), kernel_size=(11, 11), strides=(4, 4), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), padding='same'),  
    Lambda(local_response_normalization, arguments={'alpha': 1e-4, 'beta': 0.75, 'bias': 2}),  # 1st response-normalization layer
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 1st max-pooling layer


    # 2nd convolutional layer, 256 kernels of size 5 ? 5 ? 48
    Conv2D(filters=256, kernel_size=(5, 5), strides=(1, 1), activation='relu', bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), padding='same'),  
    Lambda(local_response_normalization, arguments={'alpha': 1e-4, 'beta': 0.75, 'bias': 2}),  # 2nd response-normalization layer
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 2nd max-pooling layer


    Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), activation='relu', padding='same',
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 3rd convolutional layer, 384 kernels of size 3 ? 3 ? 256


    Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), activation='relu', padding='same',
           bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 4th convolutional layer, 384 kernels of size 3 ? 3 ? 192


    Conv2D(filters=256, kernel_size=(3, 3), strides= (1, 1), activation='relu', padding='same',
           bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 5th convolutional layer, 256 kernels of size 3 ? 3 ? 192
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 3rd max-pooling layer

    Flatten(),


    Dense(units=4096, activation='relu', bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), input_shape=(32, 32, 3,)),
    Dropout(rate=0.5),


    Dense(units=4096, activation='relu', bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),
    Dropout(rate=0.5),


    Dense(units=10, bias_initializer=tf.keras.initializers.Ones(), activation='softmax',
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022))
])




In [ ]:
model.summary()

# 3. Details of Learning

It's written in the article that:
"_We used an equal learning rate for all layers, which we adjusted manually throughout training. The heuristic which we followed was to divide the learning rate by 10 when the validation error rate stopped improving with the current learning rate_"

In [ ]:
lrr = ReduceLROnPlateau(
    monitor='val_accuracy',
    patience=3
)

In [ ]:
learning_rate = 0.01
batch_size = 100
epochs1 = 10
epochs2 = 25

In [ ]:
model.compile(
    loss = tf.keras.losses.categorical_crossentropy, 
    optimizer = tfa.optimizers.SGDW(
        learning_rate=learning_rate, weight_decay = 0.0005, momentum=0.9, nesterov=True
    ) , 
    metrics=['accuracy']
)

In [ ]:
#Training the model
model_history = model.fit_generator(
    train_generator.flow(X_train, y_train, batch_size=batch_size), 
    epochs = epochs1, 
    steps_per_epoch = X_train.shape[0]//batch_size, 
    validation_data = val_generator.flow(X_val, y_val, batch_size=batch_size), 
    validation_steps = len(val_generator.flow(X_val, y_val, batch_size=batch_size)), 
    callbacks = [lrr], verbose=1)

## Other versions

After following the paper, the result seems to be not good. So we made a few adjustments. We used Batchnormalization instead of Local Response Normalization and epochs = 25

In [ ]:

model2 = Sequential([
    # 1st convolutional layer, 96 kernels of size 11?11?3 with a stride of 4 pixels
    Conv2D(filters=96, input_shape=(32, 32, 3), kernel_size=(11, 11), strides=(4, 4), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), padding='same'),  
    BatchNormalization(), 
    Activation('relu'),
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 1st max-pooling layer


    # 2nd convolutional layer, 256 kernels of size 5 ? 5 ? 48
    Conv2D(filters=256, kernel_size=(5, 5), strides=(1, 1), bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), padding='same'),  
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 2nd max-pooling layer


    Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), padding='same',
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 3rd convolutional layer, 384 kernels of size 3 ? 3 ? 256
    BatchNormalization(),
    Activation('relu'),

    Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), padding='same',
           bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 4th convolutional layer, 384 kernels of size 3 ? 3 ? 192
    BatchNormalization(),
    Activation('relu'),

    Conv2D(filters=256, kernel_size=(3, 3), strides= (1, 1), padding='same',
           bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 5th convolutional layer, 256 kernels of size 3 ? 3 ? 192
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 3rd max-pooling layer

    Flatten(),


    Dense(units=4096, bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), input_shape=(32, 32, 3,)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(rate=0.5),


    Dense(units=4096, bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(rate=0.5),


    Dense(units=10, bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),
    BatchNormalization(),
    Activation('softmax'),
])


In [ ]:
model2.compile(
    loss = tf.keras.losses.categorical_crossentropy, 
    optimizer = tfa.optimizers.SGDW(
        learning_rate=learning_rate, weight_decay = 0.0005, momentum=0.9, nesterov=True
    ) , 
    metrics=['accuracy']
)

#Training the model
model2_history = model2.fit_generator(
    train_generator.flow(X_train, y_train, batch_size=batch_size), 
    epochs = epochs2, 
    steps_per_epoch = X_train.shape[0]//batch_size, 
    validation_data = val_generator.flow(X_val, y_val, batch_size=batch_size), 
    validation_steps = len(val_generator.flow(X_val, y_val, batch_size=batch_size)), 
    callbacks = [lrr], verbose=1)

In [ ]:
(model.predict(X_test).argmax(axis=1) == y_test.argmax(axis=1)).mean()

In [ ]:
(model2.predict(X_test).argmax(axis=1) == y_test.argmax(axis=1)).mean()

We see that after our adjustments the result are much better.

In [ ]:
model2.summary()

Try with different monitor and optimizer

In [ ]:
lrr2 = ReduceLROnPlateau(
    monitor='val_accuracy',
    patience=5, min_lr=1e-5
)

In [ ]:
epochs3 = 40

model3 = Sequential([
    # 1st convolutional layer, 96 kernels of size 11?11?3 with a stride of 4 pixels
    Conv2D(filters=96, input_shape=(32, 32, 3), kernel_size=(11, 11), strides=(4, 4), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), padding='same'),  
    BatchNormalization(), 
    Activation('relu'),
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 1st max-pooling layer


    # 2nd convolutional layer, 256 kernels of size 5 ? 5 ? 48
    Conv2D(filters=256, kernel_size=(5, 5), strides=(1, 1), bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), padding='same'),  
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 2nd max-pooling layer


    Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), padding='same',
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 3rd convolutional layer, 384 kernels of size 3 ? 3 ? 256
    BatchNormalization(),
    Activation('relu'),

    Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), padding='same',
           bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 4th convolutional layer, 384 kernels of size 3 ? 3 ? 192
    BatchNormalization(),
    Activation('relu'),

    Conv2D(filters=256, kernel_size=(3, 3), strides= (1, 1), padding='same',
           bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),  # 5th convolutional layer, 256 kernels of size 3 ? 3 ? 192
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'),  # 3rd max-pooling layer

    Flatten(),


    Dense(units=4096, bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022), input_shape=(32, 32, 3,)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(rate=0.5),


    Dense(units=4096, bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(rate=0.5),


    Dense(units=10, bias_initializer=tf.keras.initializers.Ones(), 
           kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01, seed=2022)),
    BatchNormalization(),
    Activation('softmax'),
])


In [ ]:
model3.compile(
    loss = tf.keras.losses.categorical_crossentropy, 
    optimizer = 'adam' , 
    metrics=['accuracy']
)

#Training the model
model3_history = model3.fit_generator(
    train_generator.flow(X_train, y_train, batch_size=batch_size), 
    epochs = epochs3, 
    steps_per_epoch = X_train.shape[0]//batch_size, 
    validation_data = val_generator.flow(X_val, y_val, batch_size=batch_size), 
    validation_steps = len(val_generator.flow(X_val, y_val, batch_size=batch_size)), 
    callbacks = [lrr2], verbose=1)

(model3.predict(X_test).argmax(axis=1) == y_test.argmax(axis=1)).mean()